# Translate NanoHotpotQA Dataset

This notebook downloads the English NanoHotpotQA dataset from Hugging Face (`sentence-transformers/NanoBEIR-en`) and uses an OpenAI-compatible API to translate the texts into Romanian.

In [ ]:
!pip install dotenv

In [ ]:
import os
import json
import requests
import concurrent.futures
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# Load env variables from .env file
load_dotenv()

## 1. Configuration
Ensure your API keys are loaded and set your translation parameters.

In [ ]:
API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

MODEL_NAME = "gpt-4o"
MAX_WORKERS = 4      # Number of parallel API requests
TEMPERATURE = 0.3

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ WARNING: API_KEY is not set properly!")
else:
    print("✅ API key loaded.")

## 2. Translation Functions

In [ ]:
def translate_text(text: str) -> str:
    if not text:
        return text
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system", 
                "content": "You are an expert translator from English to Romanian. Your task is to translate trivia questions and information retrieval documents naturally. Ensure that the translation is fluent, contextually accurate, and avoids word-for-word literal phrasing. Pay special attention to movie titles, book titles, and grammatical structures. Provide ONLY the final Romanian translation without any extra formatting, explanations, or quotes."
            },
            {"role": "user", "content": text}
        ],
        "stream": False,
        "temperature": TEMPERATURE
    }
    
    try:
        response = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error translating: {e}")
        return text

def process_row(row):
    original_row = dict(row)
    new_row = dict(row)
    if "text" in new_row and isinstance(new_row["text"], str):
        new_row["text"] = translate_text(new_row["text"])
    return original_row, new_row

## 3. Processing Pipeline

In [ ]:
def process_and_save(config_name: str, split_name: str, output_file_ro: str, output_file_en: str):
    print(f"\nLoading {config_name} for split {split_name}...")
    dataset = load_dataset("sentence-transformers/NanoBEIR-en", config_name, split=split_name)
    
    translated_data = []
    original_data = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Map with a tqdm progress bar
        results = list(tqdm(
            executor.map(process_row, dataset), 
            total=len(dataset), 
            desc=f"Translating {config_name}"
        ))
        
        for orig, trans in results:
            original_data.append(orig)
            translated_data.append(trans)
        
    with open(output_file_ro, "w", encoding="utf-8") as f:
        for item in translated_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✅ Saved translated {config_name} to {output_file_ro}")
    
    with open(output_file_en, "w", encoding="utf-8") as f:
        for item in original_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✅ Saved original {config_name} to {output_file_en}")

## 4. Execution

In [ ]:
os.makedirs("nanohotpotqa-datasets", exist_ok=True)

print("Starting translation pipeline...\n")

# Process queries
process_and_save(
    "queries", 
    "NanoHotpotQA", 
    "nanohotpotqa-datasets/NanoHotpotQA_ro_queries.jsonl", 
    "nanohotpotqa-datasets/NanoHotpotQA_en_queries.jsonl"
)

# Process corpus
process_and_save(
    "corpus", 
    "NanoHotpotQA", 
    "nanohotpotqa-datasets/NanoHotpotQA_ro_corpus.jsonl", 
    "nanohotpotqa-datasets/NanoHotpotQA_en_corpus.jsonl"
)

# Process qrels (no text to translate usually, but handled gracefully)
process_and_save(
    "qrels", 
    "NanoHotpotQA", 
    "nanohotpotqa-datasets/NanoHotpotQA_ro_qrels.jsonl", 
    "nanohotpotqa-datasets/NanoHotpotQA_en_qrels.jsonl"
)

print("\nAll processing complete!")